[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Winfredy/SadTalker/blob/main/quick_demo.ipynb)

### SadTalker：Learning Realistic 3D Motion Coefficients for Stylized Audio-Driven Single Image Talking Face Animation 

[arxiv](https://arxiv.org/abs/2211.12194) | [project](https://sadtalker.github.io) | [Github](https://github.com/Winfredy/SadTalker)

Wenxuan Zhang, Xiaodong Cun, Xuan Wang, Yong Zhang, Xi Shen, Yu Guo, Ying Shan, Fei Wang.

Xi'an Jiaotong University, Tencent AI Lab, Ant Group

CVPR 2023

TL;DR: A realistic and stylized talking head video generation method from a single image and audio


Installation (around 5 mins)

In [ ]:
### make sure that CUDA is available in Edit -> Nootbook settings -> GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.10 2  
!python --version  
!apt-get update
!apt install software-properties-common
!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel
!apt-get install python3-pip

print('Cloning the project and installing requirements...')
!git clone https://github.com/Winfredy/SadTalker &> /dev/null
%cd SadTalker 
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH 
!python3.10 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null  
!python3.10 -m pip install -r requirements.txt

Download models (1 mins)

In [ ]:
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

In [ ]:
# borrow from makeittalk
import ipywidgets as widgets
import glob
import matplotlib.pyplot as plt

print("Choose the image name to animate (saved in folder 'examples/'):")
img_list = glob.glob('examples/source_image/*.png')
img_list.sort()
img_list = [item.split('/')[-1].split('.')[0] for item in img_list]  # Fixed path handling
default_head_name = widgets.Dropdown(options=img_list, value='full3')

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        plt.imshow(plt.imread(f'examples/source_image/{default_head_name.value}.png'))
        plt.axis('off')
        plt.show()

Animation

In [ ]:
# Selected audio from examples/driven_audio
img = f'examples/source_image/{default_head_name.value}.png'
print(img)
!python3.10 inference.py --driven_audio ./examples/driven_audio/RD_Radio31_000.wav \
--source_image {img} \
--result_dir ./results --still --preprocess full --enhancer gfpgan

In [ ]:
# visualize code from makeittalk
from IPython.display import HTML
from base64 import b64encode
import os
import glob
import sys

# Get the last result from the directory
results = sorted(os.listdir('./results/'))

if results:
    mp4_name = glob.glob(f'./results/{results[-1]}/*.mp4')[0]

    with open(mp4_name, 'rb') as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    print(f'Display animation: {mp4_name}', file=sys.stderr)
    display(HTML(f"""
      <video width=256 controls>
            <source src="{data_url}" type="video/mp4">
      </video>
      """))
else:
    print("No results found.")
